# 정적 페이지 크롤링

## 1. 패키지 불러오기

In [ ]:
# requests는 웹 서버에 요청을 보내는 데 쓰이는 라이브러리입니다.
# BeautifulSoup는 웹 서버에서 응답으로 받은 html을 파싱하는 라이브러리입니다.
# 파싱이란 텍스트의 구조를 파악해서 분리하는 행위를 의미합니다.

!pip install BeautifulSoup4

import requests
from bs4 import BeautifulSoup

## 2. HTML 파싱하기

In [ ]:
# 웹 서버에 지금 사용 중인 OS와 브라우저 정보(USER_AGENT)를 전달합니다.
# 웹 서버에 아래 정보를 전달하는 이유는 OS와 브라우저마다 웹 서버가 응답하는 방식이 다르기 때문입니다.
# 크롬을 사용하면서 파일을 다운로드 받는 경우를 예로 들면 크롬 창 아래쪽에 다운로드 완료된 항목이 표시되지만 다른 브라우저는 표시되지 않습니다.


# user-agent는 인터넷에서 검색하시면 찾아볼 수 있습니다.
USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.4 Safari/605.1.15'
headers = { 'User-Agent':USER_AGENT }


# 접근하고자하는 URL을 입력하고 요청 보내기
# get은 HTTP 프로토콜의 method 중 하나로 서버에 리소스를 요청할 때 사용합니다.
# raise_for_status()는 응답이 잘 왔는지
url = f'https://www.musinsa.com/search/musinsa/goods?q=%ED%8B%B0%EC%85%94%EC%B8%A0&list_kind=small&sortCode=pop&sub_sort=&page=1&display_cnt=0&saleGoods=&includeSoldOut=&setupGoods=&popular=&category1DepthCode=&category2DepthCodes=&category3DepthCodes=&selectedFilters=&category1DepthName=&category2DepthName=&brandIds=&price=&colorCodes=&contentType=&styleTypes=&includeKeywords=&excludeKeywords=&originalYn=N&tags=&campaignId=&serviceType=&eventType=&type=&season=&measure=&openFilterLayout=N&selectedOrderMeasure=&shoeSizeOption=&groupSale=&d_cat_cd=&attribute=&plusDeliveryYn='
res = requests.get(url, headers=headers)


# raise_for_status는 응답의 status_code를 확인할 수 있습니다.
# 200이면 성공 400대의 숫자가 출력되면 요청이나 응답에 문제가 있는것입니다.
# 먼저 url을 확인해보세요.
print(res.raise_for_status)


# soup는 BeautifulSoup으로 HTML을 파싱한 결과물입니다.
# xml은 내부적으로 트리구조를 갖고 있는 마크업 언어입니다.
# HTML은 이런 xml에 포함되는 마크업 언어 중 하나입니다.
# lxml은 Python에서 xml을 파싱하는 패키지입니다.
soup = BeautifulSoup(res.text, 'lxml')

## 3. 한 개 상품에 접근하기

In [ ]:
# 상품 목록 페이지에서 첫 번째 상품 영역에 해당하는 요소 가져오기
# find: 조건에 맞는 첫 요소를 가져올 때 사용
# find_all: 조건에 맞는 요소를 모두 가져올 때 사용

# 첫 번째 상품
item = soup.find('li', attrs={'class':'li_box'})
print(item)

## 4. 필요한 상품정보 추출하기

In [ ]:
# 브랜드
brand_elem = item.find_all('p', attrs={'class':'item_title'})[-1]
brand_elem_a = brand_elem.find('a')
brand = brand_elem_a.get_text()

# 상품명
img_elem = item.find('img')  # 이미지를 포함하는 요소
name = img_elem['alt']
name = name.replace('/', '')  # /(슬래시) 제거

# 가격
price_elem = item.find('p', attrs={'class':'price'})
price_text = price_elem.get_text()
price_list = re.findall('\d{1,3},\d{3}', price_text)
price_org = price_list[0]
price_sale = price_list[-1]

# 상세페이지
detail_elem = item.find('a', attrs={'class':'img-block'})
detail_url = detail_elem['href']

# 상세페이지 url
res_detail = requests.get(detail_url, headers=headers)   # 상세페이지 request

print(brand)
print(name)
print(price_org)
print(price_sale)
print(detail_url)

## 5. 반복문 작성

In [ ]:
import re    # regulare expression
import os    # 수업에서는 저장경로 설정과 관련된 작업을 위해 가져오겠습니다.
import requests
from bs4 import BeautifulSoup
import urllib  # 이미지 다운로드를 위한 라이브러리입니다.

# 저장 경로를 생성하겠습니다.
save_path = "crawling_tshirts"
if not os.path.exists(save_path):
    os.makedirs(save_path)

# USER_AGENT 정의하기
USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.4 Safari/605.1.15'
headers = { 'User-Agent':USER_AGENT }

all_item_info = []
for page_id in range(1,2):
    url = f'https://www.musinsa.com/search/musinsa/goods?q=%ED%8B%B0%EC%85%94%EC%B8%A0&list_kind=small&sortCode=pop&sub_sort=&page={page_id}&display_cnt=0&saleGoods=&includeSoldOut=&setupGoods=&popular=&category1DepthCode=&category2DepthCodes=&category3DepthCodes=&selectedFilters=&category1DepthName=&category2DepthName=&brandIds=&price=&colorCodes=&contentType=&styleTypes=&includeKeywords=&excludeKeywords=&originalYn=N&tags=&campaignId=&serviceType=&eventType=&type=&season=&measure=&openFilterLayout=N&selectedOrderMeasure=&shoeSizeOption=&groupSale=&d_cat_cd=&attribute=&plusDeliveryYn='
    res = requests.get(url)   # response

    soup = BeautifulSoup(res.text, 'lxml')

    # find_all을 이용해서 페이지 내 모든 상품 (90개) 가져오기
    all_item = soup.find_all('li', attrs={'class':'li_box'})

    for idx, item in enumerate(all_item):
        item_info = []
        # 브랜드
        brand_elem = item.find_all('p', attrs={'class':'item_title'})[-1]
        brand_elem_a = brand_elem.find('a')
        # print(brand_elem_a)
        brand = brand_elem_a.get_text()

        # 상품명
        img_elem = item.find('img')  # 이미지를 포함하는 요소
        name = img_elem['alt']
        name = name.replace('/', '')  # /(슬래시) 제거

        # 가격
        price_elem = item.find('p', attrs={'class':'price'})
        price_text = price_elem.get_text()
        price_list = re.findall('[0-9,]+원', price_text)
        price_org = price_list[0]
        price_sale = price_list[-1]

        # 상세페이지
        detail_elem = item.find('a', attrs={'class':'img-block'})
        detail_url = detail_elem['href']

        # 바로 위에서 얻은 상세페이지 URL (detail_url)으로 상세페이지 html 가져와서 파싱하기
        res_detail = requests.get(detail_url, headers=headers)   # 상세페이지 request
        soup_detail = BeautifulSoup(res_detail.text, 'lxml')

        # 태그 list
        tag_list = ''
        li_elem = soup_detail.find('li', attrs={'class':'article-tag-list list'})

        if li_elem == None:   # None인 경우 패스
            pass
        else:
            a_elem_list = li_elem.find_all('a')
            for a in a_elem_list:
                tag = a.get_text()
                tag = tag.replace('#', '')
                tag_list += f'{tag}, '
            tag_list = tag_list[:-2]

        # 상품이미지
        img_url = img_elem['data-original']

        item_info.append(brand)
        item_info.append(name)
        item_info.append(price_org)
        item_info.append(price_sale)
        item_info.append(detail_url)
        item_info.append(img_url)
        item_info.append(tag_list)

        all_item_info.append(item_info)
        img_url = 'https:' + img_url
        urllib.request.urlretrieve(img_url, f'{save_path}/{page_id}_{idx+1}_{name}.jpg')

## 6. 데이터프레임 저장

In [ ]:
# 테이블을 엑셀파일 형태로 저장할건데 이런 데이터를 다룰 때 쓰는 패키지가 pandas입니다.
# pandas를 다 쓰기 번거로우니 pd라고 쓰고 가져오겠습니다.
# 위 반복문에서 완성한 all_item_info

import pandas as pd

df = pd.DataFrame(all_item_info)
print(df)

In [ ]:
# 컬럼명 바꾸고 저장하기

df = df.rename(columns = {0:'브랜드', 1:'상품명', 2:'정상가', 3:'할인가', 4:'상세페이지', 5:'이미지', 6:'속성'})
print(df)
df.to_csv(f'{save_path}/table.csv', encoding='utf-8-sig')

## 7. 가격 텍스트 처리

In [ ]:
# 테이블에서 할인가를 가져오겠습니다.

price_sale_list = df['할인가'].tolist()
print( price_sale_list )

In [ ]:
# histogram에는 숫자값이 들어가야합니다.
# 위의 텍스트에서 comma(,) 그리고 '원'을 제거하겠습니다.

price_list_new = []

for price in price_sale_list:
    price_new = price.replace(',', '').replace('원', '')
    price_int = int(price_new)
    price_list_new.append(price_int)

print( price_list_new )

## 8. 가격 정보를 히스토그램으로 시각화하기

In [ ]:
# 가격 정보를 히스토그램으로 시각화하겠습니다.

import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

plt.hist(price_list_new, bins=90)
plt.xlabel('price')
plt.ylabel('count')

plt.savefig(f'{save_path}/hist.jpg')
plt.show()

## 9. 속성 키워드 가져오기

In [ ]:
# 그래프를 그리기 위해 전체 속성 키워드를 정리하겠습니다.

attrb = df['속성'].tolist()

attrb_list = []
for one_att in attrb:

    text_split = one_att.split(', ')
    attrb_list += text_split

attrb_list.remove('')

In [ ]:
attrb

## 10. 특정 키워드 빈도 카운트하기

In [ ]:
attrb_dict = {}

for attr in attrb_list:
    if attr not in attrb_dict:   # 특정 키워드가 attrb_dict에 없으면 1로 시작하기
        attrb_dict[attr] = 1
    else:  # 특정 키워드가 attrb_dict에 있으면 1씩 더하기
        attrb_dict[attr] += 1

# 크기 순서대로 정렬하기
attrb_dict_sort = dict(sorted(attrb_dict.items(), key=lambda x: x[1], reverse=True))
print( attrb_dict_sort )

## 11. 키워드 빈도 시각화

In [ ]:
# 그래프를 그릴 때 한글폰트가 필요한데 이 폰트가 저장된 위치를 가져오겠습니다.
# NanumGothic.ttc 를 찾고 이 파일이 있는 경로를 복사하겠습니다.

from matplotlib import font_manager, rc
list_font_path = [f.fname for f in font_manager.fontManager.ttflist]

# print( list_font_path )

In [ ]:
# font의 경로를 입력하면 폰트를 읽어들이고 그래프에 한글이 표시됩니다.


font_path = "/System/Library/AssetsV2/com_apple_MobileAsset_Font7/bad9b4bf17cf1669dde54184ba4431c22dcad27b.asset/AssetData/NanumGothic.ttc"
font_name = font_manager.FontProperties(fname=font_path).get_name()

rc('font', family=font_name)
plt.rcParams['font.size'] = 12

plt.figure(figsize=(40,32))  # 그래프가 그려지는 이미지의 크기를 명시합니다.
plt.bar(attrb_dict_sort.keys(), attrb_dict_sort.values())  # 키워드와 키워드의 개수를 입력해줍니다.
plt.xticks(rotation=90)  # 각 키워드를 90도 회전시킵니다
plt.xlabel('Keyword')   # x축이 어떤 걸 나타내는지 표시합니다.
plt.ylabel('Frequency')  # y축이 어떤 걸 나타내는지 표시합니다.

plt.savefig(f'{save_path}/frequency.jpg')
plt.show()

# 동적 페이지 크롤링

## 1. 동적 페이지 크롤링에 필요한 패키지 설치

In [ ]:
# 크롤링에 필요한 패키지 설치

!pip install selenium==3.14
!pip install chromedriver_autoinstaller

## 2. 크롤링에 필요한 패키지 가져오기

In [ ]:
# 크롤링에 필요한 패키지 가져오기

import time
import urllib
from PIL import Image
from selenium import webdriver
from datetime import datetime
import chromedriver_autoinstaller
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup

## 3. 크롤링 전 설정하기

In [ ]:
# USER_AGENT는 위에서 정의했지만 다시 한번 정의하겠습니다.
# SLEEP_TIME은 웹 페이지가 로드될 때까지 기다리는 시간을 의미합니다.

USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/15.4 Safari/605.1.15'
SLEEP_TIME = 1.5

In [ ]:
# 실행시키면 웹 드라이버가 설치된 경로가 표시됩니다.

chromedriver_autoinstaller.install(True)

In [ ]:
# USER_AGENT를 입력하고 웹 드라이버를 실행하겠습니다.
# 웹 드라이버는 Selenium으로 제어할 수 있는 브라우저입니다.
# 실행하면 브라우저가 뜨면서 'Chrome is being controlled ...'라는 문구가 뜹니다.

option = Options()
option.add_argument(f'user-agent={USER_AGENT}')
driver = webdriver.Chrome(options=option)
driver.maximize_window()   # 크롬 창 최대화

## 4. 랜딩페이지 접근하기

In [ ]:
# 랜딩페이지는 사용자가 홈페이지에 접속했을 때 가장 먼저 마주하는 페이지를 의미합니다.
# 실행하면 웹 드라이버의 주소창에 주소가 입력되고 페이지로 이동합니다.
# get은 BeautifulSoup를 설명할 때 말씀드린 HTTP method의 get과 같습니다.

url_landing = 'https://www.farfetch.com/'
driver.get(url_landing)

## 5. 성별 선택하기

In [ ]:
# 랜딩페이지 --> 여성
# Women, Men, Kids 버튼을 포함하는 요소의 속성 중 class에 매핑된 텍스트 'ltr-7xs1it e1p29dpb0'를 찾습니다.
# 그 요소 아래에 있는 요소 중 태그명이 a인 것을 찾습니다.
# 해당 요소의 속성값에서 href를 찾습니다.

elem_gender = driver.find_element_by_class_name('ltr-7xs1it.e1p29dpb0')
elem_women = elem_gender.find_element_by_tag_name('a')
url_women = elem_women.get_attribute('href')

print(url_women)
driver.get(url_women)

In [ ]:
# 바로 위의 코드와 동일한 동작입니다.
# 랜딩페이지 --> 여성
# driver.page_source 으로 현재 페이지의 html을 가져옵니다.
# a 태그로 시작하는 요소 중에 'data-testid'에 '여성'이 매핑된 것을 찾습니다.
# 해당 요소에 href에 매핑된 주소와 메인 페이지의 주소를 연결한 다음 url_women으로 출력합니다.
# 그리고 해당 페이지로 이동합니다.
# 앞으로 다룰 페이지 이동은 이와 같은 방식으로 진행하겠습니다.

soup_gender = BeautifulSoup(driver.page_source, 'lxml')
elem_gender = soup_gender.find('a', attrs={'data-testid':'여성'})
url_women = 'https://www.farfetch.com' + elem_gender['href']

print(url_women)
driver.get(url_women)

## 6. 대 카테고리 접근하기

In [ ]:
# '의류'에 해당하는 url 찾기

soup_cate_ext = BeautifulSoup(driver.page_source, 'lxml')
elem_cate_ext_parent = soup_cate_ext.find('ul', attrs={'class':'ltr-bbegkv'})
elem_cate_ext = elem_cate_ext_parent.find('a', attrs={'data-nav':'Clothing'})
url_cate_ext = 'https://www.farfetch.com' + elem_cate_ext['href']
text_cate_ext = elem_cate_ext.text

print(text_cate_ext)
print(url_cate_ext)
driver.get(url_cate_ext)

## 7. 중 카테고리 접근하기

In [ ]:
# 전체 필터 클릭

driver.find_element_by_class_name('ltr-8x09gq').click()

In [ ]:
# '니트웨어'에 해당하는 url 찾기

soup_cate_med = BeautifulSoup(driver.page_source, 'lxml')
elem_cate_med_parent = soup_cate_med.find('ul', attrs={'class':'ltr-11pgjmx exfleil0'})
elem_cate_med = elem_cate_med_parent('li', attrs={'class':'ltr-1xdhyk6 eibwdr01'})
url_cate_med = 'https://www.farfetch.com' + elem_cate_med[0].find('a')['href']
text_cate_med = elem_cate_med[0]['data-testid']

print(text_cate_med)
print(url_cate_med)
driver.get(url_cate_med)

## 8. 소 카테고리 접근하기

In [ ]:
# 전체 필터 클릭

driver.find_element_by_class_name('ltr-8x09gq').click()

In [ ]:
# '가디건'에 해당하는 url 찾기

soup_cate_det = BeautifulSoup(driver.page_source, 'lxml')
elem_cate_det = soup_cate_det.find_all('li', attrs={'class':'ltr-2t55jx-Body eibwdr03'})[1]
url_cate_det = 'https://www.farfetch.com' + elem_cate_det.find('a')['href']
text_cate_det = elem_cate_det['data-testid']

print(text_cate_det)
print(url_cate_det)
driver.get(url_cate_det)

## 9. 상품목록 페이지에서 상품 url 가져오기

In [ ]:
# 웹 페이지의 높이 확인하기

page_height = driver.execute_script("return document.body.scrollHeight")
step_size = page_height / 15  # 적당한 숫자로 나누기

print(f'page_height: {page_height}, step_size: {step_size}')

In [ ]:
# 천천히 스크롤 다운

for order in range(16):
    driver.execute_script(f"window.scrollTo(0, {step_size * order});")   # 스크롤 다운
    time.sleep(SLEEP_TIME*1.5)  # 대기

In [ ]:
# 페이지의 모든 상품 정보를 불러오기

soup = BeautifulSoup(driver.page_source, 'lxml')
list_item_elem = soup.find_all('a', attrs= {'data-component':'ProductCardLink'})

# list comprehension
list_item_url = ['https://www.farfetch.com'+elem['href'] for elem in list_item_elem]
list_item_url = sorted(list_item_url)

print(len(list_item_url))

# url의 일부만 출력하겠습니다.
list_item_url[:15]

In [ ]:
# list comprehension
# 반복문을 간결하게 작성할 수 있습니다.

list_item_url = []

for elem in list_item_elem:
    url = 'https://www.farfetch.com' + elem['href']
    list_item_url.append(url)

# url의 일부만 출력하겠습니다.
list_item_url[:15]

## 10. 상세페이지 이동해서 상품 정보 가져오기

In [ ]:
# 상세페이지 url 을 이용해 이동

item_url = list_item_url[0]
driver.get(item_url)
driver.execute_script("window.scrollTo(0, 1100);")
time.sleep(SLEEP_TIME*2)

In [ ]:
# 상품 정보 가져오기

soup_detail = BeautifulSoup(driver.page_source, 'lxml')
p_elem_list = soup_detail.find_all('p', attrs={'class': 'ltr-4y8w0i-Body'})
product_id = [elem.text.split(' ')[-1] for elem in p_elem_list if '파페치 ID' in elem.text][-1]

# 상품 이미지, 브랜드
image_url = soup_detail.find('div', attrs={'class': 'ltr-bjn8wh ed0fyxo0'}).find('img')['src']
brand_name = soup_detail.find('a', attrs={'class': 'ltr-8gbn9h-Heading-HeadingBold'}).text
brand_name = brand_name.replace('/', '_')

In [ ]:
# 상품명, 설명, 가격 가져오기

elem_item_name_desc = soup_detail.find_all('div', attrs={'class': 'ltr-13pqkh2 exjav150'})[0].find_all('p')
item_name = elem_item_name_desc[0].text
item_name = item_name.replace('/', '_')

# 상품 설명
try:
    description = elem_item_name_desc[-1].get_text()
except:   # 상품 설명이 없는 경우 '' 로 대체
    description = ''

# 상품 가격
try:   # 할인가가 없는 경우
    price_org = soup_detail.find('p', attrs={'data-component': 'PriceLarge'}).text.replace('₩', '').replace(',', '')
    price = price_org
except:   # 할인가가 있는 경우
    price_org = soup_detail.find('p', attrs={'data-component': 'PriceOriginal'}).text.replace('₩', '').replace(',','')
    price = soup_detail.find('p', attrs={'data-component': 'PriceFinalLarge'}).text.replace('₩', '').replace(',','')

# 이미지 다운로드
save_path = 'crawling_farfetch'
if not os.path.exists(save_path):
    os.makedirs(save_path)
urllib.request.urlretrieve(image_url, f'./{save_path}/{order+1}_{brand_name}_{item_name}.jpg')

## 11. 출력결과 확인

In [ ]:
print( product_id )
print( item_name )
print( image_url )
print( brand_name )
print( description )
print( price_org )
print( price )